### Output Parsers in LangChain

Output parsers transform the raw output returned by a language model (AIMessage or text string) into a structured format that can be directly consumed by downstream application logic.

---

### Output Parser Comparison Matrix

| Output Parser / Method | Return Type | When to Use |
| :--- | :--- | :--- |
| StrOutputParser | Plain string (str) | Convert raw AIMessage output into plain text strings. |
| JsonOutputParser | Python dictionary (dict) | Parse JSON formatted responses into Python dictionaries. |
| PydanticOutputParser | Pydantic model instance | Validate output text against a Pydantic schema class. |
| with_structured_output() | Pydantic instance or dict | Recommended modern way to enforce structured schema responses directly from the LLM. |

---

### Core Functions of Output Parsers

* Format Instructions: Provide instructions to the LLM on how to format its response (e.g. JSON schema prompt injections).
* Response Parsing: Extract and parse raw completion strings into Python data structures (strings, dicts, Pydantic objects).
* Error Handling & Retries: Support auto-fixing parsers when model completions fail schema parsing.

### 1. Environment and Model Setup

In [14]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model("gemini-3.5-flash", model_provider="google_genai")

### 2. StrOutputParser

StrOutputParser extracts the text string directly from an AIMessage response object, simplifying downstream string processing:

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Give a 1-sentence description of {topic}.")
str_parser = StrOutputParser()

# Chain prompt, model, and StrOutputParser
chain = prompt | model | str_parser

result = chain.invoke({"topic": "Quantum Computing"})

print("Type of result:", type(result).__name__)
print("Result:\n", result)

Type of result: TextAccessor
Result:
 Quantum computing is a type of computation that leverages the principles of quantum mechanics—such as superposition and entanglement—to solve complex problems exponentially faster than traditional computers.


### 3. JsonOutputParser

JsonOutputParser instructs the LLM to output a JSON object and parses the completion directly into a Python dictionary:

In [6]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Define desired schema structure
class CountryInfo(BaseModel):
    country: str = Field(description="Name of the country")
    capital: str = Field(description="Capital city")
    population_millions: float = Field(description="Approximate population in millions")

json_parser = JsonOutputParser(pydantic_object=CountryInfo)

# Inject format instructions into prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert. Respond ONLY in valid JSON matching the specified instructions."),
    ("human", "Provide details for {country}.\n{format_instructions}")
])

json_chain = prompt | model | json_parser

result = json_chain.invoke({
    "country": "Japan",
    "format_instructions": json_parser.get_format_instructions()
})

print("Type of result:", type(result).__name__)
print("Result dictionary:", result)
print("Capital:", result.get("capital"))

Type of result: dict
Result dictionary: {'country': 'Japan', 'capital': 'Tokyo', 'population_millions': 125.1}
Capital: Tokyo


### 4. PydanticOutputParser

PydanticOutputParser validates model completion text against a Pydantic schema class and returns a validated Pydantic object instance:

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

class BookDetails(BaseModel):
    title: str = Field(description="Title of the book")
    author: str = Field(description="Author name")
    genres: List[str] = Field(description="List of genres")
    publication_year: int = Field(description="Year of first publication")

pydantic_parser = PydanticOutputParser(pydantic_object=BookDetails)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract book details accurately."),
    ("human", "Extract book info for '1984 by George Orwell published in 1949 (dystopian, political fiction)'.\n{format_instructions}")
])

#LCEL
pydantic_chain = prompt | model | pydantic_parser

book = pydantic_chain.invoke({
    "format_instructions": pydantic_parser.get_format_instructions()
})

print("Type of result:", type(book).__name__)
print("Book Title:", book.title)
print("Author:", book.author)
print("Genres:", book.genres)
print("Year:", book.publication_year)

Type of result: BookDetails
Book Title: 1984
Author: George Orwell
Genres: ['dystopian', 'political fiction']
Year: 1949


### 5. Modern Recommended Approach: with_structured_output()

While traditional output parsers inject format instructions into prompts and parse raw text strings, model.with_structured_output() binds schemas directly at the model API provider level (via tool calling or native JSON mode). This is more reliable and avoids prompt parsing failures.

In [15]:
class ProgrammingLanguage(BaseModel):
    name: str = Field(description="Language name")
    creator: str = Field(description="Name of creator or organization")
    year_created: int = Field(description="Year created")
    primary_uses: List[str] = Field(description="Primary use cases")

# Modern structured output model wrapping
structured_model = model.with_structured_output(ProgrammingLanguage)

lang_info = structured_model.invoke("Python was created by Guido van Rossum and released in 1991 for web dev, data science, and scripting.")

print("Type of result:", type(lang_info).__name__)
print("Language Name:", lang_info.name)
print("Creator:", lang_info.creator)
print("Year Created:", lang_info.year_created)
print("Primary Uses:", lang_info.primary_uses)

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 1.885478404s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi